In [ ]:
from flax import nnx
from jax import numpy as jnp
from flax.nnx import combine_masks
from jax import Array
from flax.typing import Shape
import jax

def softmax_cross_attention(q: Array, k: Array, v: Array, *, is_causal = False, mask: Array | None = None):
    T_q = q.shape[-3]
    T_k = k.shape[-3]
    d_head = q.shape[-1]
    
    _mask = jnp.tri(T_q, T_k, T_k - T_q, dtype=bool) if is_causal else jnp.ones((T_q, T_k), dtype=bool)

    if mask is not None:
        _mask = _mask & mask

    attn_scores: Array = jnp.einsum("...qhd,...khd->...hqk", q, k) / (d_head ** 0.5)
    attn_scores = jnp.where(_mask, attn_scores, -jnp.inf)

    attn_weights = nnx.softmax(attn_scores, axis=-1)

    return jnp.einsum("...hqk,...khd->...qhd", attn_weights, v)

class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, rngs: nnx.Rngs):
        assert d_model % d_head == 0
        n_head = d_model // d_head

        projection = lambda : nnx.Einsum(
            "...e,ehd->...hd",
            (d_model, n_head, d_head),
            (n_head, d_head),
            rngs=rngs
        )

        self.W_q = projection()
        self.W_k = projection()
        self.W_v = projection()

        self.W_o = nnx.Einsum(
            "...hd,hde->...e",
            (n_head, d_head, d_model),
            (d_model,),
            rngs=rngs
        )

        self.ci: None | nnx.Cache[Array] = nnx.data(None)
        self.ck: None | nnx.Cache[Array] = nnx.data(None)
        self.cv: None | nnx.Cache[Array] = nnx.data(None)

    def init_cache(self, shape: Shape):
        self.ck = nnx.Cache(jnp.zeros(shape))
        self.cv = nnx.Cache(jnp.zeros(shape))
        self.ci = nnx.Cache(jnp.array(0))

    def __call__(self, X: Array, memory: Array, *, is_causal = False, decode = False, mask: Array | None = None):
        if memory is None:
            memory = X

        q = self.W_q(X)
        k = self.W_k(memory)
        v = self.W_v(memory)

        if decode:
            assert self.ci is not None
            assert self.ck is not None
            assert self.cv is not None

            # alternate approach?
            # k = self.ck[...].at[:, :, self.ci[...], :].set(q)

            (*batch, cache_len, n_head, d_head) = self.ck.shape
            zero = jnp.array(0)
            indices = (zero,) * len(batch) + (self.ci[...], zero, zero)
            k = jax.lax.dynamic_update_slice(self.ck[...], k, indices)
            v = jax.lax.dynamic_update_slice(self.cv[...], v, indices)
            self.ci[...] += k.shape[-3]

            self.ck[...] = k
            self.cv[...] = v

            # bhqk
            _mask = jnp.broadcast_to(
                jnp.arange(0, cache_len) < self.ci[...],
                # (1, 1, 1,cache_len) what I did originally, would work where len(batch) == 1
                tuple(batch) + (1, 1, cache_len)
            )
            mask = combine_masks(_mask, mask, dtype=bool)

        y = softmax_cross_attention(q, k, v, is_causal=is_causal, mask=mask)

        return self.W_o(y)